# Progetto "Analisi di Wikipedia"

In [ ]:
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, concat_ws
from pyspark.ml.feature import StringIndexer

from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.mllib.evaluation import MulticlassMetrics


## Setup dell'ambiente **Spark**

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WikipediaProject") \
    .getOrCreate()

## Funzioni riutilizzabili per visualizzazioni
Creiamo alcune funzioni per trasformare piccoli risultati Spark in DataFrame Pandas e creare grafici leggibili



In [ ]:
def plot_bar_from_spark(spark_df, x_col, y_col, title, xlabel="", ylabel="", rotation=45):
    pdf = spark_df.toPandas()

    plt.figure(figsize=(12, 6))
    plt.bar(pdf[x_col], pdf[y_col])
    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else y_col)
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()
    plt.show()


def generate_wordcloud_for_category(df, categoria, stopwords, text_col="documents"):
    testi_categoria = df.filter(
        (df["categoria"] == categoria) & (df[text_col].isNotNull())
    ).select(text_col)

    lista_testi = [row[text_col] for row in testi_categoria.collect()]
    testo_unico = " ".join(lista_testi)

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color="white",
        stopwords=stopwords
    ).generate(testo_unico)

    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"WordCloud - {categoria}")
    plt.show()


Caricamento del dataset

In [ ]:
!wget https://proai-datasets.s3.eu-west-3.amazonaws.com/wikipedia.csv -O wikipedia.csv

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv("wikipedia.csv")

df.createOrReplaceTempView("Wikipedia")

spark.sql("SELECT * FROM Wikipedia LIMIT 5").show()

--2026-05-19 20:11:41--  https://proai-datasets.s3.eu-west-3.amazonaws.com/wikipedia.csv
Resolving proai-datasets.s3.eu-west-3.amazonaws.com (proai-datasets.s3.eu-west-3.amazonaws.com)... 3.5.204.14, 3.5.206.44
Connecting to proai-datasets.s3.eu-west-3.amazonaws.com (proai-datasets.s3.eu-west-3.amazonaws.com)|3.5.204.14|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1003477941 (957M) [text/csv]
Saving to: ‘wikipedia.csv’

wikipedia.csv       100%[===================>] 956.99M  23.9MB/s    in 36s     

2026-05-19 20:12:17 (26.8 MB/s) - ‘wikipedia.csv’ saved [1003477941/1003477941]

+---+--------------------+--------------------+--------------------+---------+
|_c0|               title|             summary|           documents|categoria|
+---+--------------------+--------------------+--------------------+---------+
|  0|           economics|economics () is a...|economics () is a...|economics|
|  1|index of economic...|this aims to be a...|this aims to be a...|e

## Esplorazione iniziale del dataset


In [ ]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- documents: string (nullable = true)
 |-- categoria: string (nullable = true)



In [ ]:
df.count()

153232

In [ ]:
spark.sql("""
SELECT DISTINCT categoria
FROM wikipedia
""").show(truncate=False)

+-----------+
|categoria  |
+-----------+
|finance    |
|medicine   |
|research   |
|technology |
|energy     |
|transport  |
|politics   |
|culture    |
|science    |
|humanities |
|economics  |
|trade      |
|sports     |
|pets       |
|engineering|
+-----------+



Il dataset contiene 153232 articoli di WIkipedia, ciascuno dei quali associato a una categoria tematica e corredato da titolo, sommario e testo completo


## Analisi descrittiva dei contenuti

### **Conteggio degli articoli presenti per ogni categoria**

In [ ]:
spark.sql("""
SELECT
    categoria,
    COUNT(*) AS numero_articoli
FROM wikipedia
GROUP BY categoria
ORDER BY numero_articoli DESC
""").show(truncate=False)

+-----------+---------------+
|categoria  |numero_articoli|
+-----------+---------------+
|politics   |11358          |
|culture    |10372          |
|science    |10236          |
|humanities |10236          |
|engineering|10220          |
|finance    |10157          |
|transport  |10130          |
|economics  |10110          |
|technology |10095          |
|medicine   |10076          |
|trade      |10068          |
|sports     |10068          |
|energy     |10046          |
|research   |10037          |
|pets       |10023          |
+-----------+---------------+



In [ ]:
 plot_bar_from_spark(
            articoli_per_categoria,
            x_col="categoria",
            y_col="numero_articoli",
            title="Numero di articoli per categoria",
            xlabel="Categoria",
            ylabel="Numero di articoli"
        )


NameError: name 'articoli_per_categoria' is not defined

Dalla distribuzione degli articoli per categoria si osserva che il dataset è abbastanza bilanciato: tutte le categorie hanno circa 10.000 articoli, con una leggera prevalenza della categoria `politics`, che contiene 11.358 articoli. La categoria meno rappresentata è `pets`, con 10.023 articoli.

### **Numero medio di parole per articolo**


In [ ]:
spark.sql("""
SELECT
    categoria,
    ROUND(AVG(SIZE(SPLIT(documents, ' '))), 2) AS media_parole
FROM wikipedia
GROUP BY categoria
ORDER BY media_parole DESC
""").show(truncate=False)

In [ ]:
plot_bar_from_spark(
            media_parole_per_categoria,
            x_col="categoria",
            y_col="media_parole",
            title="Numero medio di parole per articolo",
            xlabel="Categoria",
            ylabel="Media parole"
        )


Le categorie **finance, science e politics** sono quelle in cui gli articoli presentano una lunghezza media maggiore mentre le categorie **energy e pets** contengono gli articoli con lunghezza media minore

### **Lunghezza dell'articolo più lungo e di quello più corto per ciascuna categoria**

In [ ]:
lunghezza_articoli_categoria = spark.sql("""
SELECT
    categoria,
    MAX(SIZE(SPLIT(documents, ' '))) AS articolo_piu_lungo,
    MIN(SIZE(SPLIT(documents, ' '))) AS articolo_piu_corto
FROM wikipedia
GROUP BY categoria
ORDER BY articolo_piu_lungo DESC
""")

lunghezza_articoli_categoria.show(truncate=False)

In [ ]:
lunghezze_pd = lunghezza_articoli_categoria.toPandas()

        plt.figure(figsize=(12, 6))
        plt.bar(lunghezze_pd["categoria"], lunghezze_pd["articolo_piu_lungo"], label="Articolo più lungo")
        plt.bar(lunghezze_pd["categoria"], lunghezze_pd["articolo_piu_corto"], label="Articolo più corto")
        plt.title("Lunghezza massima e minima degli articoli per categoria")
        plt.xlabel("Categoria")
        plt.ylabel("Numero di parole")
        plt.xticks(rotation=45, ha="right")
        plt.legend()
        plt.tight_layout()
        plt.show()

Oltre al confronto tra valori minimi e massimi, visualizziamo la distribuzione della lunghezza degli articoli tramite boxplot. Questo permette di osservare rapidamente la presenza di forte variabilità e di possibili valori estremi.


In [ ]:
        lunghezze_testi = spark.sql("""
        SELECT
            categoria,
            SIZE(SPLIT(documents, ' ')) AS num_parole
        FROM wikipedia
        WHERE documents IS NOT NULL
        """)

        lunghezze_testi_pd = lunghezze_testi.toPandas()

        plt.figure(figsize=(14, 6))
        lunghezze_testi_pd.boxplot(column="num_parole", by="categoria", rot=45)
        plt.suptitle("")
        plt.title("Distribuzione della lunghezza degli articoli per categoria")
        plt.xlabel("Categoria")
        plt.ylabel("Numero di parole")
        plt.tight_layout()
        plt.show()


Dall’analisi della lunghezza degli articoli emerge una forte variabilità tra i testi: alcune categorie contengono articoli molto lunghi, mentre i valori minimi indicano la presenza di articoli estremamente brevi. Questo può suggerire la presenza di record sintetici, incompleti o comunque poco informativi.



### **Creazione di nuvole di parole rappresentative per ogni categoria**

In [ ]:
df.select("categoria").distinct().show(truncate=False)

In [ ]:
categorie = [row["categoria"] for row in df.select("categoria").distinct().collect()]
categorie

In [ ]:
stopwords = set(STOPWORDS)

stopwords.update([
    "one", "new", "first", "may", "received",
    "university", "work", "including", "also",
    "known", "became", "used", "made"
])

In [ ]:
for categoria in categorie:
    generate_wordcloud_for_category(df, categoria, stopwords)


# Sviluppo di un classificatore automatico

Controlliamo il numero di valori mancanti nelle colonne che ci servono per addestrare il classificatore


*   summary
*   documents
*   categoria





In [ ]:
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in ["summary", "documents", "categoria"]
]).show()

In [ ]:
df.filter(
    col("summary").isNull() & col("documents").isNull()
).count()

Prima della costruzione del classificatore sono stati controllati i valori mancanti nelle colonne `summary`, `documents` e `categoria`. Sono state individuate 928 righe con valori nulli sia in `summary` sia in `documents`; poiché tali record non contengono testo utile per l’addestramento del modello, sono stati rimossi dal dataset.

In [ ]:
df_ml = df.filter(
    col("summary").isNotNull() &
    col("documents").isNotNull() &
    col("categoria").isNotNull()
)

In [ ]:
df_ml.count()

Aggiungiamo al dataframe una nuova colonna di testo che unisca "summary" e "documents"

In [ ]:
df_ml = df_ml.withColumn(
    "testo",
    concat_ws(" ", col("summary"), col("documents"))
)

df_ml.select("summary", "documents", "testo", "categoria").show(3, truncate=True)

Convertiamo la colonna target da colonna testuale a colonna numerica assegnando a ogni categoria un numero

In [ ]:
label_indexer = StringIndexer(
    inputCol="categoria",
    outputCol="label"
)

In [ ]:
label_indexer_model = label_indexer.fit(df_ml)
df_ml = label_indexer_model.transform(df_ml)

In [ ]:
df_ml.select("categoria", "label").distinct().orderBy("label").show(20, truncate=False)

**Preprocessing del testo**

In Spark ML i modelli richiedono che le variabili predittive siano raccolte in un’unica colonna vettoriale, solitamente chiamata `features`. In questo caso non è necessario utilizzare `VectorAssembler`, poiché la trasformazione testuale tramite CountVectorizer e IDF produce già direttamente una colonna vettoriale utilizzabile dal classificatore.

In [ ]:
# creiamo un dataframe pulito con le sole colonne che serviranno al modello ML
df_model = df_ml.select("testo", "label")

In [ ]:
df_model.show(5, truncate=True)
df_model.printSchema()

**Dividiamo il dataset in un set di training e un set di test**

In [ ]:
train, test = df_model.randomSplit([0.8, 0.2], seed=42)

In [ ]:
print("Train:", train.count())
print("Test:", test.count())

**Creiamo una pipeline di preprocessing del testo**

La pipeline racchiude le principali fasi di preprocessing e modellazione: tokenizzazione, rimozione delle stopwords, vettorizzazione, pesatura IDF e classificazione. È stata scelta una Logistic Regression multiclass perché è un modello efficiente e adatto alla classificazione testuale con feature sparse ad alta dimensionalità, come quelle prodotte da CountVectorizer e IDF.


In [ ]:
tokenizer = RegexTokenizer(
    inputCol="testo",
    outputCol="tokens",
    pattern="\\W+",
    toLowercase=True
)

stopwords_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

count_vectorizer = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    vocabSize=10000
)

idf = IDF(
    inputCol="raw_features",
    outputCol="features"
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

pipeline = Pipeline(stages=[
    tokenizer,
    stopwords_remover,
    count_vectorizer,
    idf,
    lr
])

In [ ]:
model = pipeline.fit(train)

In [ ]:
predictions = model.transform(test)

In [ ]:
predictions.select("label", "prediction", "probability").show(10, truncate=True)

**Valutiamo il modello**

In [ ]:
prediction_and_labels = predictions.select("prediction", "label") \
    .rdd \
    .map(lambda row: (float(row["prediction"]), float(row["label"])))

metrics = MulticlassMetrics(prediction_and_labels)

print("Accuracy:", metrics.accuracy)
print("Weighted Precision:", metrics.weightedPrecision)
print("Weighted Recall:", metrics.weightedRecall)
print("Weighted F1:", metrics.weightedFMeasure())

Il modello di Logistic Regression, addestrato su rappresentazioni TF-IDF del testo, ha ottenuto performance elevate sul test set, con accuracy, weighted precision, weighted recall e weighted F1-score pari a circa 0.936. Questo indica che il contenuto testuale degli articoli contiene informazioni sufficientemente distintive per permettere una classificazione efficace delle categorie tematiche.

**Analizziamo le metriche per ciascuna classe**

In [ ]:
label_mapping = df_ml.select("categoria", "label").distinct().orderBy("label")
label_mapping.show(20, truncate=False)

In [ ]:
labels = sorted([row["label"] for row in label_mapping.collect()])

metriche_per_classe = []

for label in labels:
    metriche_per_classe.append((
        float(label),
        metrics.precision(float(label)),
        metrics.recall(float(label)),
        metrics.fMeasure(float(label))
    ))

metriche_per_classe

In [ ]:
metriche_df = spark.createDataFrame(
    metriche_per_classe,
    ["label", "precision", "recall", "f1_score"]
)

metriche_df.show(20, truncate=False)

In [ ]:
metriche_categorie = label_mapping.join(metriche_df, on="label", how="inner") \
    .orderBy("label")

metriche_categorie.show(20, truncate=False)

In [ ]:
        metriche_categorie_pd = metriche_categorie.toPandas().sort_values("f1_score", ascending=True)

        plt.figure(figsize=(12, 6))
        plt.bar(metriche_categorie_pd["categoria"], metriche_categorie_pd["f1_score"])
        plt.title("F1-score per categoria")
        plt.xlabel("Categoria")
        plt.ylabel("F1-score")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()


Per approfondire gli errori del modello viene costruita una confusion matrix, così da osservare quali categorie vengono confuse più spesso dal classificatore.


In [ ]:
        confusion_df = predictions.groupBy("label", "prediction").count()

        label_mapping_pd = label_mapping.toPandas().sort_values("label")
        labels_ordinate = label_mapping_pd["label"].tolist()
        categorie_ordinate = label_mapping_pd["categoria"].tolist()

        confusion_pd = confusion_df.toPandas()

        confusion_matrix = pd.DataFrame(
            0,
            index=labels_ordinate,
            columns=labels_ordinate
        )

        for _, row in confusion_pd.iterrows():
            confusion_matrix.loc[row["label"], row["prediction"]] = row["count"]

        plt.figure(figsize=(10, 8))
        plt.imshow(confusion_matrix.values, aspect="auto")
        plt.colorbar(label="Numero di articoli")
        plt.xticks(range(len(categorie_ordinate)), categorie_ordinate, rotation=90)
        plt.yticks(range(len(categorie_ordinate)), categorie_ordinate)
        plt.xlabel("Categoria prevista")
        plt.ylabel("Categoria reale")
        plt.title("Confusion Matrix del classificatore")
        plt.tight_layout()
        plt.show()


In [ ]:
errori_frequenti = predictions.filter(col("label") != col("prediction")) \
            .groupBy("label", "prediction") \
            .count() \
            .orderBy(col("count").desc())

        errori_frequenti.show(10, truncate=False)

L’analisi delle metriche per singola categoria mostra che il classificatore ottiene performance elevate sulla maggior parte delle classi, con F1-score spesso superiori a 0.94. Le categorie `politics`, `pets` e `sports` risultano tra le più facilmente distinguibili, probabilmente perché caratterizzate da un lessico più specifico. Al contrario, categorie come `research` e `finance` presentano valori di F1-score più bassi, suggerendo una maggiore difficoltà del modello nel distinguerle da categorie semanticamente vicine, come `science`, `medicine`, `engineering`, `economics` o `trade`.

# **Identificazione di nuovi insights**

Dall’analisi descrittiva e dalla successiva fase di classificazione automatica emergono diversi insights utili per comprendere meglio la struttura del dataset Wikipedia.

- In primo luogo, la distribuzione degli articoli tra le diverse categorie
risulta complessivamente bilanciata: la maggior parte delle categorie contiene circa 10.000 articoli, con una leggera prevalenza della categoria `politics`. Questo è un notevole vantaggio in quanto si riduce il rischio che il classificatore impari a privilegiare una classe dominante a scapito delle altre.

- L’analisi della lunghezza degli articoli mostra invece una maggiore variabilità: alcune categorie contengono articoli molto lunghi, mentre i valori minimi evidenziano la presenza di testi estremamente brevi. Questo suggerisce che, pur essendo bilanciato dal punto di vista del numero di articoli, il dataset contiene documenti con livelli di approfondimento molto diversi.

- Le WordCloud permettono di osservare le principali tendenze linguistiche associate alle diverse categorie. Alcune classi presentano un vocabolario più specifico e riconoscibile, mentre altre condividono termini simili con categorie affini. Per esempio, categorie come `science`, `research`, `medicine` ed `engineering` possono presentare sovrapposizioni di parole, così come `finance`, `economics` e `trade`.

- Questa sovrapposizione si riflette anche nei risultati del classificatore: le metriche per singola categoria mostrano infatti performance molto elevate per classi come `politics`, `sports` e `pets`, che sembrano avere un lessico più distintivo. Al contrario, categorie come `research` e `finance` ottengono F1-score più bassi rispetto alla media, suggerendo una maggiore difficoltà del modello nel distinguerle da classi semanticamente vicine.

Nel complesso, gli insights ottenuti indicano che il contenuto testuale degli articoli è fortemente informativo per la categorizzazione automatica, ma alcune aree tematiche presentano confini linguistici meno netti. </br>Questo aspetto potrebbe essere utile per Wikimedia/Wikidata Insights al fine di migliorare l’organizzazione dei contenuti, individuare categorie potenzialmente sovrapposte e ottimizzare futuri sistemi automatici di classificazione.

## Conclusioni

Il progetto ha permesso di analizzare un dataset di articoli Wikipedia con l’obiettivo di comprenderne la struttura, individuare caratteristiche descrittive rilevanti e sviluppare un sistema automatico di classificazione delle categorie tematiche.

Nella prima parte del lavoro è stata condotta un’analisi esplorativa dei contenuti, utilizzando Spark SQL per calcolare la distribuzione degli articoli per categoria, il numero medio di parole per articolo e la lunghezza minima e massima dei documenti all’interno di ciascuna classe. L’analisi ha mostrato che il dataset è complessivamente bilanciato dal punto di vista della numerosità delle categorie, con circa 10.000 articoli per la maggior parte delle classi, ma presenta una maggiore variabilità nella lunghezza dei testi, con alcuni articoli molto estesi e altri estremamente brevi.

Le WordCloud generate per ciascuna categoria hanno permesso di osservare le principali tendenze linguistiche associate ai diversi argomenti. Da questa analisi è emerso che alcune categorie presentano un lessico più distintivo, mentre altre condividono termini e concetti simili, rendendo potenzialmente più complessa la loro separazione automatica.

Nella seconda parte del progetto è stato sviluppato un classificatore automatico basato su una Pipeline Spark ML. Le colonne `summary` e `documents` sono state combinate in un’unica variabile testuale, successivamente sottoposta a tokenizzazione, rimozione delle stopwords, vettorizzazione tramite CountVectorizer e pesatura tramite IDF. Il modello finale, basato su Logistic Regression multiclass, ha ottenuto performance elevate sul test set, con accuracy, weighted precision, weighted recall e weighted F1-score pari a circa 0.936.

L’analisi delle metriche per singola categoria ha mostrato che il classificatore funziona molto bene per classi con vocabolario più specifico, come `politics`, `sports` e `pets`, mentre incontra maggiori difficoltà su categorie semanticamente più vicine, come `research` e `finance`.

Nel complesso, il progetto dimostra che il contenuto testuale degli articoli Wikipedia contiene informazioni sufficientemente informative per supportare un sistema efficace di classificazione automatica.